In [ ]:
import hashlib
import urllib.parse

def generate_youdao_dict_params(text, le='en'):
    """
    生成有道词典API请求参数
    
    Args:
        text (str): 要查询的文本
        le (str): 语言方向，默认'en'
    
    Returns:
        dict: 包含签名参数的字典
    """
    
    def md5_hash(data):
        """MD5哈希函数"""
        if not isinstance(data, str):
            data = str(data)
        return hashlib.md5(data.encode('utf-8')).hexdigest()
    
    # 常量定义
    v = "webdict"
    salt_key = "Mk6hqtUp33DGGtoS63tTJbMUYjRrG1Lu"
    
    # 计算时间参数
    time = len(f"{text}{v}") % 10
    
    # 生成第一个MD5
    r = f"{text}{v}"
    o = md5_hash(r)
    
    # 生成签名的MD5
    n = f"web{text}{time}{salt_key}{o}"
    f = md5_hash(n)
    
    # 构造参数字典
    d = {
        'q': text,
        'le': le,
        't': time,
        'client': 'web',
        'sign': f,
        'keyfrom': v
    }
    
    return d


In [ ]:
import requests
import json
# from crawler_strategy import SeleniumCrawlerStrategy
# from action import By, Do, SeleniumAction
import time
from lxml import etree
REQUEST_HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 '
                  'Safari/537.36',
    'Content-Type': 'application/x-www-form-urlencoded',
    "Connection": "close"
}
def get_cn_name(en_name):
    # base_url = 'https://corp.dict.cn/search?q=%s' % en_name
    # resp = requests.get(base_url, headers=REQUEST_HEADERS)
    # e = etree.HTML(resp.text)
    # return ''.join(e.xpath('//*[@id="content"]/div[1]/div[1]/div[3]/ul/li[1]/strong/text()'))
    base_url = "https://dict.youdao.com/jsonapi_s?doctype=json&jsonversion=4"
    forms = generate_youdao_dict_params(en_name)
    # forms = {
    #     "q": en_name,
    #     "le": "en",
    #     "t": 4,
    #     "client": "web",
    #     "sign": "a9372983f37e643c500fa26f738cc8f5",
    #     "keyfrom": "webdict"
    # }
    rsp = requests.post(url=base_url, headers=REQUEST_HEADERS, data=forms)
    data = json.loads(rsp.text).get('web_trans')
    if data:
        trans = data.get("web-translation")
        if trans:
            trans = trans[0].get("trans")
            return trans[0].get("value", "")
    return ""

In [ ]:
REQUEST_HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 '
                  'Safari/537.36',
    'Content-Type': 'application/json;charset=UTF-8',
    'Referer':"https://ieeexplore.ieee.org/xpl/conhome/10946287/proceeding", 
    "Connection": "close"
}
def get_HPCA(url):
    response = requests.get(url=url, headers=REQUEST_HEADERS)
    data = json.loads(response.text).get("records")
    return [
        {"year": item.get("issues")[0].get("year"),
         "publicationNumber": item.get("publicationNumber"),
         "issueNumber": item.get("issues")[0].get("issueNumber")
        } for item in data]
ieee_hpca_url = 'https://ieeexplore.ieee.org/rest/publication/conhome/metadata?parentId=1000335'
hpca_lists = get_HPCA(url=ieee_hpca_url)
hpca_lists

In [ ]:
def get_proceeding_list(punumber, isnumber, pagenum):
    proceeding_url = f'https://ieeexplore.ieee.org/rest/search/pub/{punumber}/issue/{isnumber}/toc'
    REQUEST_HEADERS = {
        'Host': 'ieeexplore.ieee.org',
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36',
        'Accept': 'application/json, text/plain, */*',
        'Content-Type': 'application/json;charset=UTF-8',
        # 'Referer': 'https://ieeexplore.ieee.org/xpl/conhome/10476359/proceeding?sortType=vol-only-seq&isnumber=10476395&pageNumber=3',
        'Origin': 'https://ieeexplore.ieee.org',
        'Connection': 'close'
    }
    payload = {
        "isnumber": isnumber,
        "pageNumber": pagenum,
        "punumber": punumber,
        "sortType": "vol-only-seq"
    }
    response = requests.post(proceeding_url, headers=REQUEST_HEADERS, data=json.dumps(payload))
    return json.loads(response.text).get("records")

In [ ]:
import re
def get_article(id):
    article_url = f'https://ieeexplore.ieee.org/document/{id}'
    REQUEST_HEADERS = {
        'Host': 'ieeexplore.ieee.org',
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
        'Content-Type': 'text/html;charset=UTF-8',
        # 'Referer': 'https://ieeexplore.ieee.org/xpl/conhome/10476359/proceeding?sortType=vol-only-seq&isnumber=10476395&pageNumber=3',
        'Origin': 'https://ieeexplore.ieee.org',
        'Connection': 'close'
    }
    response = requests.get(url=article_url, headers=REQUEST_HEADERS)
    response.status_code
    content = response.text
    pattern = r'xplGlobal\.document\.metadata\s*=\s*({.*?});'
    match = re.search(pattern, content, re.DOTALL)
    if match:
        metadata_json = match.group(1)
        metadata_data = json.loads(metadata_json)
        return metadata_data.get("authors")
    return None

In [ ]:
import csv

In [ ]:
for item in hpca_lists[1:]:
    pubnumber = item.get("publicationNumber")
    isnumber = item.get("issueNumber")
    year = item.get("year")
    for page in range(1, 20):
        records = get_proceeding_list(punumber=pubnumber, isnumber=isnumber, pagenum=page)
        data_saved = []
        if records is None:
            break
        for record in records:
            if record.get("abstract") is None:
                continue
            article_number = record.get("articleNumber")
            article_title = record.get("articleTitle")
            article_info = get_article(id=article_number)
            if article_info is None:
                continue
            tmp_schools = []
            for seq, info in enumerate(article_info):
                schools = [school.lower() for school in ''.join(info.get('affiliation', '')).split(',') 
                           if 'university' in school.lower()]
                for school in schools:
                    if school in tmp_schools:
                        break
                    tmp_schools.append(school)
                    school_name = get_cn_name(school) or school
                    author_name = info.get("name")
                    data_saved.append(
                        ['HPCA', year, school, school_name, seq + 1, article_title, page, article_number]
                    )
                    break
                time.sleep(0.1)
            print(f"🍎 完成第{page} 页 {article_number} 文章的抓取")
            time.sleep(0.2)
        print(f"✅ 已完成 {year} 年 第{page} 页文章的抓取, 共计 {len(data_saved)}")
        
        with open(f'HPCA_{year}.csv', 'a', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerows(data_saved)
        time.sleep(0.5)

In [2]:
from run_crawler import RunCrawler
from extraction import PageLinkExtraction

crawler = RunCrawler()
crawler.run(url="https://jwc.cqu.edu.cn/index/tzgg.htm",
            extraction_strategy=PageLinkExtraction())

<class 'NoneType'>
[Logger]🍎 Launching LocalRequestsCrawlerStrategy.


'{\n    "url": "https://jwc.cqu.edu.cn/index/tzgg.htm",\n    "title": "通知公告-重庆大学本科教学信息网",\n    "keywords": "重庆大学本科教学信息网.重大本科教学信息网,本科教学信息网,通知公告",\n    "description": "",\n    "res": {\n        "urls": {\n            "0": [\n                "https://jwc.cqu.edu.cn/info/1084/5932.htm",\n                "https://jwc.cqu.edu.cn/info/1080/5930.htm",\n                "https://jwc.cqu.edu.cn/info/1080/5927.htm",\n                "https://jwc.cqu.edu.cn/info/1084/5972.htm",\n                "https://jwc.cqu.edu.cn/info/1084/5968.htm",\n                "https://jwc.cqu.edu.cn/info/1080/5967.htm",\n                "https://jwc.cqu.edu.cn/info/1084/5966.htm",\n                "https://jwc.cqu.edu.cn/info/1084/5962.htm",\n                "https://jwc.cqu.edu.cn/info/1084/5953.htm",\n                "https://jwc.cqu.edu.cn/info/1084/5952.htm",\n                "https://jwc.cqu.edu.cn/info/1084/5931.htm",\n                "https://jwc.cqu.edu.cn/info/1084/5929.htm",\n                "https://jwc.cqu.

In [ ]:
from run_crawler import RunCrawler
from run_crawler import *

crawl_strategy = RequestsCrawlerStrategy()
crawler = RunCrawler(crawler_strategy=crawl_strategy)
crawler.run(...) # 通过内部的crawler_strategy.crawl()执行
# 更新策略
crawler.set_crawler_strategy(SeleniumCrawlerStrategy(...))
crawler.run(...)    # 调用新对象的.crawl()

: 